# Exploratory Data Analysis
### 1. Dataset Overview

The Online Retail II dataset contains historical transaction records from a UK-based online retailer. Each row represents an individual transaction line, including invoice, product, quantity, price, customer, and geographic information.

The following exploration examines the dataset's structure, completeness, and characteristics before data preparation.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [0]:
PROJECT_ROOT = Path.cwd().parents[0]

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_online_retail_II.csv"

In [0]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"[+] Loaded Dataset shape: {df.shape[0]:,} rows and {df.shape[1]} columns")

In [0]:
df.head(5)

### 2. Structural and Missing-Value Analysis

In [0]:
print("="*50)
print("\tSTRUCTURAL DATA TYPES & NULLS")
print("="*50)
df.info()

In [0]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

missing_summary

### 2.2 Numerical Summary

In [0]:
print("="*50)
print("\tSUMMARY STATISTICS")
print("="*50)
display(df[['Quantity', 'Price']].describe().T)

### 3. Business Data Dictionary

In [0]:
# Business Definitions for Each Column
# The Below dictionary has the following format
# |Label|Units|Definition|
business_definitions = {
    "Invoice": ("Invoice Number", "ID", "Unique transaction identifier (6-7 characters). Starts with '4' or '5' for standard sales, 'C' for cancellations, and 'A' for administrative adjustments."),
    "StockCode": ("Stock Code", "SKU", "Product or administrative item code. Primarily 5-digit numeric SKUs, alongside operational/shipping codes (e.g., 'POST', 'DOT') and discount flags."),
    "Description": ("Product Description", "Text", "Descriptive name for products, operational fees, or vouchers. Contains historical missing values and warehouse logging noise."),
    "Quantity": ("Quantity Sold", "Count", "Number of units ordered per transaction line. Negative values indicate product returns or reversals."),
    "InvoiceDate": ("Invoice Timestamp", "Datetime", "Exact date and time when the transaction was generated."),
    "Price": ("Unit Price", "GBP (£)", "Product price per unit stated in British Pounds Sterling."),
    "Customer ID": ("Customer ID", "ID", "Unique 5-digit customer identifier. Contains 107,927 missing entries."),
    "Country": ("Country", "Text", "Geographic region where the transaction or customer is located (40 unique countries).")
}

In [0]:
# |Feature Name|Business Label|Data Type|unit|Null count|Unique values|Description|
dict_list = []

for col in df.columns:
    label, unit, definition = business_definitions.get(col, (col, "N/A", "No Definition"))
    dict_list.append({
        "Feature Name": col,
        "Business Label": label,
        "Data Type": str(df[col].dtype),
        "Unit": unit,
        "Null Count": df[col].isnull().sum(),
        "Unique Values": df[col].nunique(),
        "Description": definition
    })

dict_df = pd.DataFrame(dict_list)
display(dict_df)

In [0]:
REPORTS_DIR_PATH = PROJECT_ROOT / "reports"
DICTIONARY_PATH = REPORTS_DIR_PATH / "DataDictionary.csv"
REPORTS_DIR_PATH.mkdir(parents=True, exist_ok=True)

dict_df.to_csv(DICTIONARY_PATH, index=False)
print(f"\n[+]Exported Business Data Dictionary artifact to: {DICTIONARY_PATH}")

### 4. Invoice & Transaction analysis

The Invoice ID starts with 4 or 5; if it starts with a C its a cancelled order, and if it starts with an A it signifies adminstrative adjustments.

In [0]:
cancel_df = df.query('Invoice.str.startswith("C")')
adjust_df = df.query('Invoice.str.startswith("A")')

print(f"Number of cancelled orders: {cancel_df.shape[0]}")
print(f"Number of adminstrative adjustment orders: {adjust_df.shape[0]}")

print(f"\nTotal number of negative quantity orders: {df[df['Quantity']<0].shape[0]}")
print(f"Total number of negative price orders: {df[df['Price']<0].shape[0]}")

print(f"\nTotal number of negative quantity and C coded Invoice orders: {cancel_df[cancel_df['Quantity']<0].shape[0]}")
print(f"Total number of negative price and C coded Invoice orders: {cancel_df[cancel_df['Price']<0].shape[0]}")

print(f"\nTotal number of negative quantity and A coded Invoice orders: {adjust_df[adjust_df['Quantity']<0].shape[0]}")
print(f"Total number of negative price and A coded Invoice orders: {adjust_df[adjust_df['Price']<0].shape[0]}")



In [0]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df

In [0]:
cancel_df[cancel_df['Quantity'] > 0]

In [0]:
invoices_start = df['Invoice'].str[0].unique()
starter_counts = df['Invoice'].str[0].value_counts()

print("## Invoice Data EDA")
print("\n1. Invoice starting characters and counts:")
print(starter_counts)

print("\n2. Entries starting with A:")
print("Although quantities are not negative, prices are negative.")
display(df[df['Invoice'].str[0] == 'A'])

print("\n3. Code C entries with negative quantities:")
c_negative = df[
    (df['Invoice'].str[0] == 'C') &
    (df['Quantity'] < 0)
]
print(f"Number of entries: {c_negative.shape[0]}")
print("The entries have been checked for negative prices.")
display(c_negative)

print("\n4. Total number of negative quantities:")
print(df[df['Quantity'] < 0].shape[0])

print("\n5. Total number of negative prices:")
print(df[df['Price'] < 0].shape[0])
print("The negative prices appear to be associated with invoice code A.")
display(df[df['Price'] < 0])

print("\n6. Negative quantities excluding code C:")
other_negative = df[
    (df['Invoice'].str[0] != 'C') &
    (df['Quantity'] < 0)
]
print(f"Number of entries: {other_negative.shape[0]}")
print("Check whether these entries have invoice codes starting with 4 or 5 and prices of zero.")
display(other_negative)

In [0]:
df[(df['Invoice'].str[0] != "C") & (df['Quantity'] < 0)].shape[0]

In [0]:
print("="*50)
print("\tUNIVARIATE SKEWNESS DIAGNOSTICS")
print("="*50)

numeric_df = df.select_dtypes(include=[np.number])

skewness_series = numeric_df.skew().sort_values(ascending=False)

skew_summary = pd.DataFrame({
    "Feature": skewness_series.index,
    "Skewness Coefficient": skewness_series.values,
    "Diagnosis": ["Extremely Right-Skewed" if abs(s) > 3.0 else "Moderately Skewed" if abs(s) > 1.0 else "Unskewed" for s in skewness_series.values],
    "Pipeline Action": ["Isolate returns/adjustments; aggregate for RFM" for s in skewness_series.values]
})

display(skew_summary)

In [0]:
print("Invoice prefixes:")
print(df["Invoice"].astype(str).str[0].value_counts())

print("\nNegative quantities by invoice prefix:")
print(pd.crosstab(
    df["Invoice"].astype(str).str[0],
    df["Quantity"] < 0,
    margins=True
))

In [0]:
display(
    df.loc[df["Price"] <= 0]
    .sort_values("Price")
    .head(15)
)

In [0]:
display(
    df.loc[df.duplicated(keep=False)]
    .sort_values(["Invoice", "StockCode", "InvoiceDate"])
    .head(20)
)

In [0]:
duplicate_mask = df.duplicated(keep=False)

print("Rows involved in duplicate groups:", duplicate_mask.sum())
print("Exact duplicate rows after the first occurrence:",
      df.duplicated().sum())

print("\nDuplicate groups by invoice:")
print(
    df.loc[duplicate_mask]
    .groupby("Invoice")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

In [0]:
duplicate_mask = df.duplicated(keep=False)

duplicate_groups = (
    df.loc[duplicate_mask]
    .groupby(
        ["Invoice", "StockCode", "Description", "Quantity",
         "InvoiceDate", "Price", "Customer ID", "Country"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

display(duplicate_groups.head(10).reset_index(name="Occurrences"))

In [0]:
print("Invoice prefix counts:")
print(df["Invoice"].astype(str).str.startswith("C").sum(),
      "cancellation-prefixed rows")

print("\nRows with negative quantity:", (df["Quantity"] < 0).sum())
print("Rows with zero quantity:", (df["Quantity"] == 0).sum())
print("Rows with negative price:", (df["Price"] < 0).sum())
print("Rows with zero price:", (df["Price"] == 0).sum())

print("\nInvoice prefix x price sign:")
print(pd.crosstab(
    df["Invoice"].astype(str).str[0],
    pd.cut(
        df["Price"],
        bins=[float("-inf"), 0, 0.000001, float("inf")],
        labels=["Negative", "Zero", "Positive"]
    )
))